In [ ]:
from pathlib import Path

import torch
from PIL import Image
from transformers import AutoProcessor, MllamaForConditionalGeneration

In [ ]:
# Update this path to your local Llama model
model_id = "/path/to/Llama-3.2-11B-Vision-Instruct"
# Update this path to your test image
imageName = "sample_data/synthetic_invoice_014.png"

print("🔧 Loading Llama-3.2-Vision model...")
model = MllamaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(model_id)

print("📷 Loading image...")
image = Image.open(imageName)
print(f"✅ Image loaded: {image.size}")

In [ ]:
# Visual Question Answering - ask a simple question about the image
question = "How much did Jessica pay?"

# Create message structure for Llama
messageDataStructure = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {
                "type": "text",
                "text": question,
            },
        ],
    }
]

print(f"❓ Question: {question}")
print("🤖 Generating response with Llama-3.2-Vision...")

In [ ]:
# Process the input
textInput = processor.apply_chat_template(
    messageDataStructure, add_generation_prompt=True
)
inputs = processor(image, textInput, return_tensors="pt").to(model.device)

# Generate response
output = model.generate(**inputs, max_new_tokens=2000)
generatedOutput = processor.decode(output[0])

print("✅ Response generated successfully!")
print("\n" + "="*60)
print("LLAMA-3.2-VISION RESPONSE:")
print("="*60)
print(generatedOutput)
print("="*60)

In [ ]:
# Optional: Save the response to a file
output_path = Path("llama_vqa_output.txt")

with output_path.open("w", encoding="utf-8") as text_file:
    text_file.write(generatedOutput)

print(f"✅ Response saved to: {output_path}")
print(f"📄 File size: {output_path.stat().st_size} bytes")